# WellCo Churn Prediction — Phase 3: Modelling

Trains a LightGBM binary classifier on the engineered feature frame.

**Key design choices:**
- **Outreach as a training feature**: outreach is a post-observation treatment applied on Jul 15 (after the observation window, before churn measurement). Including it in training lets the model quantify its effect. At scoring time we set `outreach=0` for all test members — they have not yet received outreach — so scores represent *pre-outreach churn risk*, exactly what the prioritisation list should reflect.
- **Primary metric: AUC-ROC** — the task is ranking members by churn risk, not predicting a threshold. AUC directly measures ranking quality. PR-AUC (average precision) is reported alongside to account for the 20% class imbalance.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, roc_curve
)

train = pd.read_parquet('train_features.parquet')
test  = pd.read_parquet('test_features.parquet')

TARGET   = 'churn'
DROP     = ['member_id', 'churn', 'outreach']
FEAT_COLS = [c for c in train.columns if c not in DROP]
ALL_COLS  = FEAT_COLS + ['outreach']   # outreach included during training

X = train[ALL_COLS]
y = train[TARGET]

print('Train shape:', X.shape)
print('Feature count:', len(ALL_COLS))
print('Churn rate:', y.mean().round(4))
print('Outreach rate:', train['outreach'].mean().round(4))

## 1. Cross-validation

5-fold stratified CV. `is_unbalance=True` upweights the minority (churn) class; early stopping on each fold's validation AUC.

In [ ]:
LGB_PARAMS = dict(
    objective        = 'binary',
    metric           = 'auc',
    n_estimators     = 1000,
    learning_rate    = 0.05,
    num_leaves       = 31,
    min_child_samples= 20,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    is_unbalance     = True,
    random_state     = 42,
    verbose          = -1,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train))
fold_results = []
best_iters = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMClassifier(**LGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False),
                   lgb.log_evaluation(-1)]
    )
    best_iters.append(model.best_iteration_)
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]

    auc = roc_auc_score(y_val, oof_preds[val_idx])
    ap  = average_precision_score(y_val, oof_preds[val_idx])
    ll  = log_loss(y_val, oof_preds[val_idx])
    fold_results.append(dict(fold=fold+1, auc=auc, ap=ap, logloss=ll,
                             best_iter=model.best_iteration_))
    print(f'Fold {fold+1}: AUC={auc:.4f}  PR-AUC={ap:.4f}  LogLoss={ll:.4f}  '
          f'(best iter={model.best_iteration_})')

res = pd.DataFrame(fold_results)
print(f'\nMean AUC:     {res.auc.mean():.4f} ± {res.auc.std():.4f}')
print(f'Mean PR-AUC:  {res.ap.mean():.4f} ± {res.ap.std():.4f}')
print(f'Mean LogLoss: {res.logloss.mean():.4f} ± {res.logloss.std():.4f}')
print(f'\nOOF AUC:    {roc_auc_score(y, oof_preds):.4f}')
print(f'OOF PR-AUC: {average_precision_score(y, oof_preds):.4f}')

## 2. Final model — train on all data

Use mean best iteration from CV as `n_estimators` (no validation set needed; early stopping already chose the depth).

In [ ]:
final_params = dict(LGB_PARAMS)
final_params['n_estimators'] = int(np.mean(best_iters))
print('Final model n_estimators:', final_params['n_estimators'])

final_model = lgb.LGBMClassifier(**final_params)
final_model.fit(X, y, callbacks=[lgb.log_evaluation(-1)])
print('Final model trained.')

## 3. Evaluation plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- ROC curve ---
fpr, tpr, _ = roc_curve(y, oof_preds)
axes[0].plot(fpr, tpr, color='steelblue',
             label=f'OOF AUC = {roc_auc_score(y, oof_preds):.4f}')
axes[0].plot([0, 1], [0, 1], '--', color='gray', linewidth=0.8)
axes[0].set_title('ROC Curve (OOF)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

# --- Score distribution by churn label ---
scores_df = pd.DataFrame({'score': oof_preds, 'churn': y})
for label, color, name in [(0,'steelblue','No churn'), (1,'tomato','Churn')]:
    axes[1].hist(scores_df[scores_df['churn']==label]['score'],
                 bins=40, alpha=0.6, color=color, label=name, density=True)
axes[1].set_title('OOF Score Distribution')
axes[1].set_xlabel('Predicted probability')
axes[1].set_ylabel('Density')
axes[1].legend()

# --- CV fold AUC bar chart ---
axes[2].bar(res['fold'], res['auc'], color='steelblue', alpha=0.8)
axes[2].axhline(res['auc'].mean(), color='tomato', linestyle='--',
                label=f'Mean = {res["auc"].mean():.4f}')
axes[2].set_title('AUC by CV Fold')
axes[2].set_xlabel('Fold')
axes[2].set_ylabel('AUC')
axes[2].set_ylim(0.5, 1.0)
axes[2].legend()

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=120)
plt.show()

## 4. Feature importance

In [ ]:
feat_imp = (pd.DataFrame({'feature': ALL_COLS,
                           'importance': final_model.feature_importances_})
              .sort_values('importance', ascending=True))

fig, ax = plt.subplots(figsize=(7, 11))
colors = ['tomato' if f == 'outreach' else 'steelblue' for f in feat_imp['feature']]
ax.barh(feat_imp['feature'], feat_imp['importance'], color=colors)
ax.set_title('LightGBM Feature Importance (split count)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.show()

print(feat_imp.sort_values('importance', ascending=False).to_string(index=False))

## 5. Outreach effect analysis

Quantify the treatment effect: for each training member, compare predicted churn probability at `outreach=1` vs `outreach=0`. This validates that the model correctly captures outreach as protective, and informs the cost-benefit framing in Phase 4.

In [ ]:
X_outreach_0 = X.copy(); X_outreach_0['outreach'] = 0
X_outreach_1 = X.copy(); X_outreach_1['outreach'] = 1

p0 = final_model.predict_proba(X_outreach_0)[:, 1]
p1 = final_model.predict_proba(X_outreach_1)[:, 1]
effect = p0 - p1   # positive = outreach reduces churn risk

print('Outreach effect on predicted churn probability (p_no_outreach - p_outreach):')
print(f'  Mean reduction:   {effect.mean():.4f}')
print(f'  Median reduction: {np.median(effect):.4f}')
print(f'  Min / Max:        {effect.min():.4f} / {effect.max():.4f}')
print(f'  Members helped (effect > 0): {(effect > 0).mean():.1%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(effect, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(effect.mean(), color='tomato', linestyle='--',
                label=f'Mean = {effect.mean():.3f}')
axes[0].set_title('Distribution of outreach effect\n(p_no_outreach − p_outreach)')
axes[0].set_xlabel('Risk reduction')
axes[0].legend()

# Effect vs base risk (outreach=0 score)
axes[1].scatter(p0, effect, alpha=0.05, s=5, color='steelblue')
axes[1].set_title('Outreach effect vs base churn risk')
axes[1].set_xlabel('Predicted churn risk (outreach=0)')
axes[1].set_ylabel('Risk reduction from outreach')

plt.tight_layout()
plt.savefig('outreach_effect.png', dpi=120)
plt.show()

## 6. Score test members & save predictions

In [ ]:
X_test = test[FEAT_COLS].copy()
X_test['outreach'] = 0   # test members have not yet received outreach

test_scores = final_model.predict_proba(X_test)[:, 1]

predictions = pd.DataFrame({
    'member_id':   test['member_id'],
    'churn_score': test_scores,
    'rank':        pd.Series(test_scores).rank(ascending=False, method='first').astype(int)
}).sort_values('rank').reset_index(drop=True)

print('Test score distribution:')
print(predictions['churn_score'].describe().round(4))
print('\nTop 10 highest-risk members:')
print(predictions.head(10).to_string(index=False))

predictions.to_csv('predictions.csv', index=False)
print('\nSaved predictions.csv')